<a href="https://colab.research.google.com/github/AetherLewis/ollama-webUI/blob/main/Running-Ollama-on-Google-Colab-Through-Pinggy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install pciutils and Ollama
!sudo apt-get install -y pciutils
!curl https://ollama.ai/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 51 not upgraded.
Need to get 343 kB of archives.
After this operation, 1,581 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Fetched 343 kB in 0s (4,731 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: 

In [2]:
# Install pinggy
!pip install pinggy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 121.8 MB/s eta 0:00:00


In [3]:
import subprocess
import os
import time
import requests

def start_ollama_server():
    ollama_path = '/usr/local/bin/ollama'
    install_script_url = 'https://ollama.ai/install.sh'
    install_script_path = '/tmp/install_ollama.sh'

    if not os.path.exists(ollama_path):
        print(f"Ollama executable not found at {ollama_path}. Attempting to install Ollama and its dependencies...")

        try:
            # Step 0: Ensure zstd dependency is installed
            print("Checking and installing zstd dependency...")
            subprocess.run(['sudo', 'apt-get', 'update', '-y'], check=True, capture_output=True, text=True)
            subprocess.run(['sudo', 'apt-get', 'install', '-y', 'zstd'], check=True, capture_output=True, text=True)
            print("zstd installed successfully.")

            # Step 1: Download the Ollama installation script
            print(f"Downloading Ollama installation script from {install_script_url}...")
            response = requests.get(install_script_url, stream=True)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

            with open(install_script_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            os.chmod(install_script_path, 0o755) # Make the script executable
            print("Ollama installation script downloaded successfully.")

            # Step 2: Execute the installation script with an updated PATH
            print("Executing Ollama installation script...")
            # Ensure /usr/bin is in PATH for the script to find zstd
            env = os.environ.copy()
            if '/usr/bin' not in env.get('PATH', ''):
                env['PATH'] = '/usr/bin:' + env['PATH']

            install_process = subprocess.run(
                ['bash', install_script_path],
                check=True,
                capture_output=True,
                text=True,
                env=env # Pass the modified environment
            )
            print("Ollama installation stdout:")
            print(install_process.stdout)
            if install_process.stderr:
                print("Ollama installation stderr:")
                print(install_process.stderr)
            print("Ollama installation command executed. Waiting a few seconds...")
            time.sleep(10) # Give it some time for paths and daemon to settle

            if not os.path.exists(ollama_path):
                print(f"Installation failed. Ollama still not found at {ollama_path}. Please check the installation logs above.")
                return
            print("Ollama installed successfully.")

        except requests.exceptions.RequestException as e:
            print(f"Error downloading Ollama installation script: {e}")
            return
        except subprocess.CalledProcessError as e:
            print(f"Error during Ollama installation script execution (return code: {e.returncode}):")
            print(f"Stdout: {e.stdout}")
            print(f"Stderr: {e.stderr}")
            return
        except Exception as e:
            print(f"An unexpected error occurred during installation: {e}")
            return

    # Kill any existing Ollama server processes to ensure a clean restart with new env vars
    try:
        print("Checking for and terminating existing Ollama server processes...")
        kill_process = subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True, text=True)
        if kill_process.returncode == 0:
            print("Existing Ollama server processes terminated.")
            time.sleep(5) # Give it some time to fully shut down
        else:
            print("No existing Ollama server processes found to terminate or pkill failed.")
    except Exception as e:
        print(f"Error terminating existing Ollama server processes: {e}")


    # Check if Ollama server is already running, if not, start it
    try:
        # Use 'pgrep -f' to find processes with 'ollama serve' in their command line
        # This is a more robust way to check if the server is already active
        subprocess.run(['pgrep', '-f', 'ollama serve'], check=True, capture_output=True)
        print("Ollama server appears to be already running.")
    except subprocess.CalledProcessError:
        print("Ollama server not found running. Starting it...")
        # Start Ollama server in a new process so it doesn't block the notebook
        # Set OLLAMA_HOST to 0.0.0.0 to allow external connections
        env = os.environ.copy()
        env['OLLAMA_HOST'] = '0.0.0.0'
        subprocess.Popen([ollama_path, 'serve'], env=env)
        print("🚀 Ollama server launched successfully!")
    except Exception as e:
        print(f"Could not check for running Ollama server: {e}. Attempting to launch anyway.")
        env = os.environ.copy()
        env['OLLAMA_HOST'] = '0.0.0.0'
        subprocess.Popen([ollama_path, 'serve'], env=env)
        print("🚀 Ollama server launched successfully!")

start_ollama_server()

Ollama executable not found at /usr/local/bin/ollama. Attempting to install Ollama and its dependencies...
Checking and installing zstd dependency...
zstd installed successfully.
Ollama installation script downloaded successfully.
Executing Ollama installation script...
Ollama installation stdout:

Ollama installation stderr:
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                         
##O#-#                                                                        

                                                                           0.0%
                                                                           0.4%
                                                                           0.9%
#                                                                          1.7%
##                                                                         2.8%
##                                

In [4]:
# Check if Ollama is listening on port 11434
def check_ollama_port(port='11434'):
    try:
        output = subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'],
                              capture_output=True, text=True).stdout
        if f":{port} (LISTEN)" in output:
            print(f"✅ Ollama is actively listening on port {port}")
        else:
            print(f"⚠️ Ollama not detected on port {port}")
    except Exception as e:
        print(f"❌ Error checking port: {e}")
check_ollama_port()

✅ Ollama is actively listening on port 11434


In [5]:
# Start a pinggy tunnel to expose the Ollama API
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:11434",
    headermodification=["u:Host:localhost:11434"]
)

print(f"Tunnel1 started - URLs: {tunnel1.urls}")

Tunnel1 started - URLs: ['http://dqxil-34-124-227-235.run.pinggy-free.link', 'https://dqxil-34-124-227-235.run.pinggy-free.link']


In [6]:
import subprocess
import os

model_name = "qwen2.5:14b"
ollama_path = '/usr/local/bin/ollama' # Assuming Ollama is installed here

try:
    print(f"Attempting to pull model: {model_name}\n")
    # Run ollama pull command and capture output
    # Using subprocess.run for better output capture and error handling
    result = subprocess.run(
        [ollama_path, 'pull', model_name],
        check=True,  # Raise CalledProcessError if the command returns a non-zero exit code
        capture_output=True,
        text=True,
        env=os.environ # Pass current environment to ensure ollama can find its dependencies
    )
    print("Ollama pull stdout:")
    print(result.stdout)
    if result.stderr:
        print("Ollama pull stderr:")
        print(result.stderr)
    print(f"\nModel '{model_name}' pulled successfully.")

except subprocess.CalledProcessError as e:
    print(f"\nError pulling model '{model_name}' (return code: {e.returncode}):")
    print(f"Stdout:\n{e.stdout}")
    print(f"Stderr:\n{e.stderr}")
except FileNotFoundError:
    print(f"\nError: Ollama executable not found at {ollama_path}. Please ensure Ollama is installed and its path is correct.")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

Attempting to pull model: qwen2.5:14b

Ollama pull stdout:

Ollama pull stderr:
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 2049f5674b1e:   0% ▕                  ▏  14 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏  73 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏ 112 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 160 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 200 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 226 MB/9.0 GB                  pulling manifes

In [7]:
# Make a request to the Ollama API through the tunnel
import requests
import json

# Dynamically retrieve the HTTPS tunnel URL
tunnel_url = tunnel1.urls[1]

response = requests.post(f"{tunnel_url}/api/generate",
                        json={
                            "model": "qwen2.5:14b",
                            "prompt": "Hello, how are you?",
                            "stream": False
                        })

print(f"Status Code: {response.status_code}")
print("Response Text:")
print(response.text)

Status Code: 200
Response Text:
{"model":"qwen2.5:14b","created_at":"2026-05-20T19:09:15.79757361Z","response":"Hello! I'm an artificial intelligence and don't have feelings, but I'm here to help you with any questions or tasks you might have. How can I assist you today?","done":true,"done_reason":"stop","context":[151644,8948,198,2610,525,1207,16948,11,3465,553,54364,14817,13,1446,525,264,10950,17847,13,151645,198,151644,872,198,9707,11,1246,525,498,30,151645,198,151644,77091,198,9707,0,358,2776,458,20443,11229,323,1513,944,614,15650,11,714,358,2776,1588,311,1492,498,448,894,4755,476,9079,498,2578,614,13,2585,646,358,7789,498,3351,30],"total_duration":122695848588,"load_duration":119738797308,"prompt_eval_count":35,"prompt_eval_duration":180841934,"eval_count":37,"eval_duration":2656294138}


In [8]:
# Install Open WebUI
!pip install open-webui

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.4/151.4 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
# Start a pinggy tunnel for Open WebUI
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:8000",
)

print(f"Tunnel1 started - URLs: {tunnel1.urls}")

Tunnel1 started - URLs: ['http://plcxq-34-124-227-235.run.pinggy-free.link', 'https://plcxq-34-124-227-235.run.pinggy-free.link']


## Access Open WebUI

Click on the following URL to access the Open WebUI interface:

```
https://plcxq-34-124-227-235.run.pinggy-free.link
```

(This URL was provided by `tunnel1.urls` in the previous cell's output.)

In [ ]:
# Start the Open WebUI server on port 8000
!open-webui serve --port 8000

Loading WEBUI_SECRET_KEY from file, not provided as an environment variable.
Generating a new secret key and saving it to /content/.webui_secret_key
Loading WEBUI_SECRET_KEY from /content/.webui_secret_key
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 7e5b5dc7342b, init
INFO  [alembic.runtime.migration] Running upgrade 7e5b5dc7342b -> ca81bd47c050, Add config table
INFO  [alembic.runtime.migration] Running upgrade ca81bd47c050 -> c0fbf31ca0db, Update file table
INFO  [alembic.runtime.migration] Running upgrade c0fbf31ca0db -> 6a39f3d8e55c, Add knowledge table
Creating knowledge table
Migrating data from document table to knowledge table
INFO  [alembic.runtime.migration] Running upgrade 6a39f3d8e55c -> 242a2047eae0, Update chat table
Converting 'chat' column to JSON
Renaming 'chat' column to 'old_chat'
Adding new 'chat' column of type JSON
Dropping 'old